# 🎬 IndoClip Pro — Google Colab GPU Backend (GitHub Edition)

Jalankan backend **IndoClip Pro** langsung dari repository GitHub Anda di Google Colab untuk memanfaatkan **NVIDIA T4 GPU (16GB VRAM)** dan internet datacenter gratis!

### 📌 Langkah Persiapan di Colab:
1. Di menu atas Colab, klik **Runtime** -> **Change runtime type** -> pilih **T4 GPU** -> klik **Save**.
2. Masukkan URL GitHub Anda di **Sel 2**, lalu jalankan semua sel secara berurutan.

### Sel 1: Verifikasi GPU NVIDIA

In [ ]:
!nvidia-smi

### Sel 2: Masukkan URL GitHub Repo & Clone Otomatis
Ganti link repository di bawah ini dengan link repository GitHub IndoClip Anda.

In [ ]:
import os

# Masukkan link GitHub repository Anda di sini:
GITHUB_REPO_URL = "https://github.com/nausamah27/IndoClip.git" # @param {type:"string"}

repo_name = GITHUB_REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
os.chdir("/content")

if not os.path.exists(repo_name):
    print(f"Sedang meng-clone repository {GITHUB_REPO_URL}...")
    !git clone {GITHUB_REPO_URL}
else:
    print(f"Repository sudah ada, memperbarui dari GitHub...")
    os.chdir(f"/content/{repo_name}")
    !git pull
    os.chdir("/content")

backend_dir = f"/content/{repo_name}/backend"
if os.path.exists(backend_dir):
    os.chdir(backend_dir)
    print(f"\n✅ Berhasil masuk ke direktori backend: {os.getcwd()}")
else:
    print(f"\n⚠️ Folder backend tidak ditemukan di {backend_dir}, periksa struktur repository Anda.")

### Sel 3: Install Dependensi Python untuk GPU

In [ ]:
!pip install -q fastapi uvicorn yt-dlp faster-whisper google-genai "mediapipe>=0.10.14" opencv-python numpy scipy python-dotenv sse-starlette aiofiles requests

### Sel 4: Download Cloudflare Tunnel (Tunneling Publik Gratis)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

### Sel 5: Jalankan Backend IndoClip Pro & Dapatkan URL Publik HTTPS
Salin URL yang berakhiran `.trycloudflare.com` di bawah ini, lalu masukkan ke Vercel Environment Variable `NEXT_PUBLIC_API_URL`.

In [ ]:
import subprocess
import time
import re

# Jalankan backend FastAPI di background port 8000
backend_proc = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Jalankan Cloudflare Tunnel ke port 8000
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

print("\n=======================================================")
print("   Mencari URL Publik Cloudflare Tunnel...")
print("=======================================================")

tunnel_url = None
while True:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print(f"\n🚀 URL BACKEND ANDA TELAH AKTIF:\n{tunnel_url}\n")
            print("👉 Buka website Vercel Anda, klik tombol 'Backend: Offline/Set URL' di pojok kanan atas,")
            print(f"   lalu tempelkan URL ini: {tunnel_url}")
            print("   (Atau masukkan ke Vercel Environment Variables: NEXT_PUBLIC_API_URL)")
            break

try:
    backend_proc.wait()
except KeyboardInterrupt:
    print("Backend dihentikan.")
    backend_proc.kill()
    tunnel_proc.kill()